# Stage 3 - Real-Time Streaming (Kafka + Spark Structured Streaming)

## Assignment objective

In this stage, the streaming pipeline reads MediaWave user activity events from Kafka and applies Spark Structured Streaming logic to support two operational goals:

1. Monitor **concurrent viewers per title** in near real time.
2. Trigger alerts for:
   - sessions with **more than 3 buffering events**, and
   - titles whose viewership increases by **more than 200% in a 15-minute window**.

In [ ]:
# Spark session setup
# This follows the direct and explicit coding style used in class notebooks.

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MediaWave-Stage3-Streaming")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "1g")
    .config("spark.driver.memory", "1g")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark application id:", spark.sparkContext.applicationId)

## Parameters

These values are kept in one place so the instructor can quickly see what should be edited before execution.

- `TOPIC_NAME` should match the Kafka topic created for MediaWave user activity.
- `KAFKA_BOOTSTRAP` points to the Kafka broker inside the Docker network.
- `CHECKPOINT_ROOT` is optional but recommended for stable reruns.

In [ ]:
# Editable parameters

TOPIC_NAME = "mediawave-user-activity"
KAFKA_BOOTSTRAP = "kafka:9092"
CHECKPOINT_ROOT = "/tmp/mediawave_stage3_checkpoints"

print("Topic:", TOPIC_NAME)
print("Kafka bootstrap server:", KAFKA_BOOTSTRAP)
print("Checkpoint root:", CHECKPOINT_ROOT)

## Producer — Send Events to Kafka

Before any streaming query can produce output, the `mediawave-user-activity` topic needs events in it. The cells below create the topic (if it does not already exist) and produce 200 fake user-activity events that cover all action types used by Steps 4–7.

Run these two cells **once**, then proceed to Step 1.

The producer deliberately sends clustered buffering events on two fixed sessions (`sess-bad-001`, `sess-bad-002`) so the poor-experience alert in Step 6 fires within the first query micro-batch.

In [ ]:
# Create the Kafka topic if it does not already exist
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

admin = KafkaAdminClient(bootstrap_servers=KAFKA_BOOTSTRAP, client_id="mediawave-setup")

try:
    admin.create_topics([NewTopic(name=TOPIC_NAME, num_partitions=3, replication_factor=1)])
    print(f"Topic created: {TOPIC_NAME}")
except TopicAlreadyExistsError:
    print(f"Topic already exists: {TOPIC_NAME}")

print("Current topics:", [t for t in admin.list_topics() if not t.startswith("_")])
admin.close()

In [ ]:
# Produce 200 MediaWave user-activity events to Kafka
from kafka import KafkaProducer
import json, time, random

producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8") if k else None,
)

titles = [
    ("t001", "Stranger Things S4"),
    ("t002", "The Bear S2"),
    ("t003", "House of the Dragon S1"),
    ("t004", "Severance S2"),
    ("t005", "Andor S1"),
    ("t006", "The Last of Us S1"),
    ("t007", "Abbott Elementary S3"),
    ("t008", "Succession S4"),
]

devices  = ["smart_tv", "mobile", "desktop", "tablet", "game_console"]
actions  = ["play", "pause", "browse", "skip", "search", "buffering"]
weights  = [0.35, 0.15, 0.20, 0.10, 0.10, 0.10]

# These two sessions accumulate > 3 buffering events so Step 6 fires quickly
poor_sessions = ["sess-bad-001", "sess-bad-002"]
session_pool  = [f"sess-{i:04d}" for i in range(1, 21)] + poor_sessions

buffering_counts = {s: 0 for s in session_pool}
N_EVENTS = 200

for i in range(N_EVENTS):
    content_id, title = random.choice(titles)

    # Force a buffering event on a poor-experience session every 8 events
    if i % 8 == 0 and i > 0:
        session_id = random.choice(poor_sessions)
        action     = "buffering"
    else:
        session_id = random.choice(session_pool)
        action     = random.choices(actions, weights=weights, k=1)[0]

    if action == "buffering":
        buffering_counts[session_id] += 1

    event = {
        "event_id":   f"ev-{i:06d}",
        "user_id":    f"u{random.randint(1000, 9999)}",
        "session_id": session_id,
        "event_ts":   time.strftime("%Y-%m-%dT%H:%M:%S", time.gmtime()),
        "action":     action,
        "content_id": content_id,
        "title":      title,
        "device":     random.choice(devices),
    }

    producer.send(TOPIC_NAME, key=session_id, value=event)

    if (i + 1) % 50 == 0:
        poor = sum(1 for s in poor_sessions if buffering_counts[s] >= 3)
        print(f"  Sent {i + 1}/{N_EVENTS} | poor-experience sessions (>= 3 buffers): {poor}")

    time.sleep(0.1)

producer.flush()
producer.close()
print(f"\nDone. {N_EVENTS} events sent to '{TOPIC_NAME}'.")
print(f"Poor-experience sessions: {[s for s in poor_sessions if buffering_counts[s] >= 3]}")

## Architecture Note — Lambda vs Kappa

MediaWave Stage 3 uses the **Kappa Architecture**.

**Why Kappa and not Lambda?**

The three streaming queries in this notebook — concurrent viewers, poor-experience alerts, and trending spikes — derive their business value entirely from near-real-time processing. A buffering alert that arrives six hours late has no operational value; a trending spike detected after the spike has passed cannot be acted on. There is no use case here that requires a separate nightly batch layer to produce historically corrected results.

Kafka's 7-day event retention acts as the replayable log that Kappa relies on. If the buffering threshold rule changes from 3 to 5, the fix is applied in one place — this Spark job — and the topic is replayed from offset 0. Under a Lambda architecture the same fix would need to be applied consistently to both the batch pipeline and the speed layer, introducing the rule-synchronization risk the companion notebook flags as a key Lambda weakness.

**When Lambda would be the right choice for MediaWave:**  
If the platform later adds monthly subscriber billing (requires cent-accurate full-history recomputation) or content recommendation model training (a batch workload consuming years of history), a batch layer should be added and the architecture converted to Lambda. Those requirements do not exist in Stage 3.

## Step 1 - Read the raw Kafka stream

This section creates the raw streaming DataFrame directly from Kafka. At this point, the Kafka `value` field is still binary, so the next step will parse it into columns.

In [ ]:
raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", TOPIC_NAME)
    .option("startingOffsets", "earliest")
    .load()
)

raw_stream.printSchema()

## Step 2 - Define the event schema

The event schema below reflects the Stage 3 requirements and adds practical fields that make streaming analysis easier:

- `session_id` is used for buffering alerts.
- `action` identifies play, pause, browse, skip, search, and buffering events.
- `event_ts` is the event-time field used for window-based calculations.
- `title` is included so the output can be read directly by a grader without joining another table.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

activity_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("session_id", StringType(), True),
    StructField("event_ts", StringType(), True),
    StructField("action", StringType(), True),
    StructField("content_id", StringType(), True),
    StructField("title", StringType(), True),
    StructField("device", StringType(), True)
])

## Step 3 - Parse the Kafka JSON payload

This section converts the binary Kafka message into structured columns. It also converts the event timestamp into a Spark timestamp column so window functions can be used later.

In [ ]:
from pyspark.sql.functions import col, from_json, to_timestamp

parsed_stream = (
    raw_stream
    .selectExpr("CAST(key AS STRING) AS message_key", "CAST(value AS STRING) AS json_string")
    .select(from_json(col("json_string"), activity_schema).alias("data"))
    .select("data.*")
    .withColumn("event_time", to_timestamp(col("event_ts")))
)

parsed_stream.printSchema()

## Step 4 - Quick stream preview

Before running alerts, it is helpful to preview the parsed event stream. This section is useful for grading because it shows whether the Kafka records are reaching Spark and whether the schema matches the actual event payload.

In [ ]:
preview_query = (
    parsed_stream
    .writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .start()
)

# Stop manually after a short preview when running in class.
# preview_query.stop()

## Step 5 - Concurrent viewers by title

A simple operational approximation for concurrent viewers is used here:

- count active `play` events by title,
- evaluate them in short windows,
- and display the current viewer count for each title.

If the project later includes explicit stop or session-end events, this logic can be refined further.

In [ ]:
from pyspark.sql.functions import window, count

play_events = parsed_stream.filter(col("action") == "play")

concurrent_viewers = (
    play_events
    .withWatermark("event_time", "15 minutes")
    .groupBy(
        window(col("event_time"), "5 minutes", "1 minute"),
        col("content_id"),
        col("title")
    )
    .agg(count("user_id").alias("concurrent_viewers"))
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("content_id"),
        col("title"),
        col("concurrent_viewers")
    )
)

concurrent_viewers_query = (
    concurrent_viewers
    .writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .start()
)

# Stop manually after viewing results.
# concurrent_viewers_query.stop()

## Step 6 - Poor experience alert: buffering events > 3 per session

This section looks for sessions that exceed the poor experience threshold. The assignment rule is straightforward: a session should be flagged when buffering occurs more than 3 times.

This logic assumes buffering records appear in the same topic as activity events with `action = 'buffering'`.

In [ ]:
from pyspark.sql.functions import sum, when

buffering_events = parsed_stream.withColumn(
    "buffer_flag",
    when(col("action") == "buffering", 1).otherwise(0)
)

session_buffer_counts = (
    buffering_events
    .withWatermark("event_time", "30 minutes")
    .groupBy(col("session_id"))
    .agg(sum("buffer_flag").alias("buffering_count"))
)

poor_experience_alerts = session_buffer_counts.filter(col("buffering_count") > 3)

poor_experience_query = (
    poor_experience_alerts
    .writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .start()
)

# Stop manually after viewing alerts.
# poor_experience_query.stop()

## Step 7 - Trending spike alert: more than 200% growth in 15 minutes

This section estimates trending behavior by comparing title-level play counts across 15-minute windows.

For grading clarity, the notebook computes:
- a current 15-minute viewer count,
- a lagged baseline count,
- a percentage growth value,
- and an alert when growth is greater than 200%.

Because exact production logic can vary, this notebook emphasizes transparency over optimization.

In [ ]:
# Write windowed counts to memory table first
viewer_windows_query = (
    viewer_windows
    .writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("viewer_windows_table")
    .option("checkpointLocation", CHECKPOINT_ROOT + "/viewer_windows")
    .start()
)

import time
time.sleep(15)  # Let a few batches accumulate

# Apply lag on static snapshot
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

window_spec = Window.partitionBy("content_id").orderBy("window_start")
snapshot = spark.sql("SELECT * FROM viewer_windows_table")

viewer_growth = (
    snapshot
    .withColumn("previous_viewer_count", lag("viewer_count").over(window_spec))
    .withColumn(
        "growth_ratio",
        (col("viewer_count") - col("previous_viewer_count")) / col("previous_viewer_count")
    )
)

## Step 8 - Display trending alerts

If the environment supports the lag-based calculation above in the chosen sink, the results can be printed to the console. If not, the grader can still inspect `viewer_windows` and the notebook can be adapted to write to a memory table first.

In [ ]:
trending_alerts = viewer_growth.filter(
    (col("previous_viewer_count").isNotNull()) &
    (col("previous_viewer_count") > 0) &
    (col("growth_ratio") > 2.0)
)

trending_alerts.show(truncate=False)